This hybrid document similarity pipeline enables flexible querying of a document set using either single words or full sentences, leveraging two pre-trained models for optimal results. For single-word queries like “drug,” it employs the Word2Vec model (Google News-trained) to generate 300-dimensional word embeddings, comparing the query vector to each token in preprocessed documents—tokenized and stripped of stopwords and punctuation using NLTK—and ranks documents by their maximum cosine similarity, ensuring word-level precision consistent with earlier outputs. For multi-word queries like “pain relief medication,” it switches to the Sentence-Transformers model (all-MiniLM-L6-v2), embedding entire documents and the query as 384-dimensional vectors to capture broader semantic context, then ranks them by cosine similarity. The pipeline automatically detects the query type (single-word vs. multi-word) and applies the appropriate model, sorting results by similarity and displaying those above a 0.5 threshold, offering a scalable, pre-trained solution that adapts to new documents without retraining.

In [29]:
!pip install nltk gensim sentence-transformers

In [20]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
import gensim.downloader as api
from sentence_transformers import SentenceTransformer
from numpy import dot
from numpy.linalg import norm

# Download NLTK resources
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')
nltk.download('stopwords')

# Load pre-trained models
word2vec_model = api.load("word2vec-google-news-300")  # Word-level
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')  # Sentence-level

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [21]:
documents = [
    "The patient was prescribed a new drug for pain relief.",
    "Pharmaceutical companies are developing Rx medications.",
    "He got addicted to oxycodone after surgery.",
    "This is a random document about coding and Python.",
    "Drugs like heroin are illegal in many states."
]

In [27]:
stop_words = set(stopwords.words('english') + list(string.punctuation))

def preprocess(text):
    tokens = word_tokenize(text.lower())
    return [token for token in tokens if token not in stop_words and token.isalnum()]

processed_docs = [preprocess(doc) for doc in documents]

def cosine_similarity(vec1, vec2):
    return dot(vec1, vec2) / (norm(vec1) * norm(vec2))

def search_documents(query):
    is_single_word = len(query.split()) == 1

    if is_single_word and query in word2vec_model:
        query_vector = word2vec_model[query]
        doc_scores = []
        for i, doc_tokens in enumerate(processed_docs):
            max_similarity = 0
            for token in doc_tokens:
                if token in word2vec_model:
                    similarity = cosine_similarity(query_vector, word2vec_model[token])
                    max_similarity = max(max_similarity, similarity)
            doc_scores.append((max_similarity, i))
    else:
        query_vector = sentence_model.encode(query)
        doc_vectors = sentence_model.encode(documents)
        doc_scores = [(cosine_similarity(query_vector, doc_vector), i)
                      for i, doc_vector in enumerate(doc_vectors)]

    doc_scores.sort(reverse=True)
    print(f"Documents similar to '{query}':")
    for score, idx in doc_scores:
        if score > 0.5:
            print(f"Score: {score:.3f} - {documents[idx]}")

In [33]:
# Test with single-word query
print("Single-word query:\n")
search_documents("drug")

# Test with multi-word query
print("\nMulti-word query:\n")
search_documents("pain relief medication")

Single-word query:

Documents similar to 'drug':
Score: 1.000 - The patient was prescribed a new drug for pain relief.
Score: 0.849 - Drugs like heroin are illegal in many states.
Score: 0.558 - He got addicted to oxycodone after surgery.
Score: 0.537 - Pharmaceutical companies are developing Rx medications.

Multi-word query:

Documents similar to 'pain relief medication':
Score: 0.721 - The patient was prescribed a new drug for pain relief.
